# Reusable model packs — v3 startup

Use the same approach as your uploaded **H3_Model_Packs Data Sets.ipynb**:

1. Run **Image_Studio_Model_Packs.ipynb** once with **Internet ON** and **Accelerator: None**, leaving `PACK = 'standard'`.
2. Wait for **PACK COMPLETE**, then **Save Version with output files** and keep it private.
3. Here, choose **Add Input → Your Work / Notebook Output Files** and attach that saved version. Enable **Internet + T4 GPU**, then run the studio cell.

No named Dataset or Kaggle API token is needed for this saved-output approach. `REQUIRE_ATTACHED_MODELS = True` stops startup if weights are missing. Attached files are checksum-verified and read directly without another full model copy. Software setup and the studio tunnel still use Internet.

For FLUX3, build a separate **flux3** pack with accepted Hugging Face terms and Kaggle Secret HF_TOKEN, attach its saved output alongside the standard pack, and enable `ENABLE_FLUX3`. The renderer needs no Hugging Face token once the complete weights are attached.

The generation/upscaling workflows are unchanged from v2. Earlier notebooks remain available.

---

# Image Studio — Kaggle v3

Single-cell ComfyUI image studio, built for Shaun's Kaggle T4 workflow.

1. Enable **Internet** and a **T4 GPU** in Kaggle settings.
2. Run the cell below once; open its **OPEN IMAGE STUDIO** URL.
3. In ComfyUI's workflow menu, open **Image Studio - Create**, **Restyle**, **Inpaint**, or **Detail**, or an upscaling workflow. Change the prompt and press **Run**. No image starts automatically.
4. Images and logs are in `/kaggle/working/image-studio`. **Save Version with outputs** before ending your session. PNG metadata retains the generation graph and seed; drag a PNG into ComfyUI to restore it.

**Create:** 1024×1024, 30 steps, DPM++ 2M Karras, CFG 5.5. Try 832×1216 portrait or 1216×832 landscape, batch 1. These are starting settings, not a universal optimum.

**Restyle:** upload a source with dimensions around 1 megapixel; denoise 0.6 balances source structure and change. Lower preserves more structure. **Inpaint:** upload, right-click the image node → Open in MaskEditor, paint the area to change, save the mask and run. Uses SDXL latent-noise masking, not a dedicated inpainting checkpoint; seams and identity changes remain possible.

**Detail:** 1024px generation followed by 1.5× latent enlargement and a 0.3-denoise detail pass. This is generative detail, not faithful super-resolution; it can change faces/text. It is slower and uses more VRAM. If memory fails, use Create or reduce the selected resolution explicitly. No automatic quality reductions occur.

**Model:** [Juggernaut XL v9](https://huggingface.co/RunDiffusion/Juggernaut-XL-v9), about 7.1 GB, verified with the model file's published SHA256. Read its model card and underlying SDXL license before commercial use. This is a practical SDXL quality option for T4, not a claim to be the best model for every image. One T4 runs the graph; a second T4 is not pooled VRAM. No paid image API is used.

The checkpoint is reused during the session. Use the CPU model pack builder to save it once; attached inputs are discovered and checksum-verified. Model weights/runtime live in `/kaggle/temp`, not in saved image outputs. Dependencies require Internet even with an attached checkpoint. ComfyUI and cloudflared are pinned; Kaggle's Torch stack is preserved with constraints. Future Kaggle image changes may need a compatibility update.

**Validation limit:** see the research and validation report linked beside the notebook for completed local checks. Kaggle GPU generation, output quality, T4 speed, and public tunnel reliability remain unverified. If setup fails, retain `comfy.log` and `tunnel.log`. Keep the startup cell running; stopping it stops its own server and tunnel. Quick Tunnel URLs grant access to anyone holding the URL.

## Upscaling added in v2

**Upscale 2x / Upscale 4x:** upload an existing image in the Load Image node and press Run. Uses the official RealESRGAN x4plus checkpoint and ComfyUI's tiled model upscaling. The 2x option runs the 4x model and downsamples with Lanczos. This can still change small details; inspect faces and text. A 1024×1024 input becomes 2048×2048 or 4096×4096 respectively.

**Upscale Creative:** upload an image, replace the positive prompt with a description of that image, then Run. Uses Ultimate SD Upscale with 768px tiles, batch 1, 20 steps and denoise 0.20; starts at 2x. It reuses your SDXL checkpoint. Expect more compute and possible invented texture, face changes and seams. It is a connected graph with the upstream Ultimate SD Upscale node; the pipeline is not hidden inside a new wrapper.

**Restore SeedVR2:** native ComfyUI SeedVR2 3B INT8 workflow with FP16 compute, tiled VAE encoding/decoding, one sampling step and a 2x starting size. This is generative restoration and can change detail. The diffusion transformer is not spatially tiled: VAE tiling alone does not guarantee a large target fits in VRAM. Start with a source at 1024px or below. Increase the multiplier only after a successful run. No automatic render or hidden resolution reduction.

The standard model pack contains about 11.1 GB of model files. Set `LOAD_SDXL = False` to skip the 7.1 GB SDXL download and show only the enlargement/restoration workflows; set `ENABLE_SEEDVR2 = False` to skip its approximately 4.0 GB. RealESRGAN adds about 67 MB. First setup needs additional dependency space. Attached Kaggle inputs with matching filenames and SHA256 values are reused. An existing v1 checkpoint is also reused within the same session.

Stop the older studio cell before starting v2. The new notebook has a separate runtime, current pinned ComfyUI, and model-specific workflows. Earlier v1 remains available. Keep your original images; upscaling cannot recover unknowable ground truth.

## Optional FLUX 3 Klein generation

Set `ENABLE_FLUX3 = True` only if you want this model. Accept the [FLUX 3 Klein Comfy model terms](https://huggingface.co/black-forest-labs/flux-3-image-klein-comfy), create a Hugging Face read token with access to that model, and store it in Kaggle **Secrets → HF_TOKEN**, enabled for this notebook. Never paste a token into a prompt or notebook code. You can set `LOAD_SDXL = False` to avoid downloading the older generator too.

The **Create FLUX3** graph expands ComfyUI's official template into visible ordinary nodes. It retains the required Qwen3.5 prompt upsampler, shows the enhanced prompt, uses the INT8 distilled generator/text encoder, a W4A8 prompt enhancer and tiled VAE decode. Starts at 1024×1024, 16 steps, Euler/simple, CFG 1. The extra models total about 17.75 GB. This profile is experimental here: graph loading is checked, but gated model download, T4 inference, speed and visual quality are unverified. An 8 GB RTX 3070 claim from upstream does not establish T4 performance. T4 uses FP16 compute and compatible INT8/dequantization paths; no FP8/NVFP4 hardware acceleration is assumed. The weights use the FLUX Kommunity License, distinct from the code's Apache-2.0 license.


In [ ]:
# Single-cell startup. No image generation starts automatically.
from pathlib import Path
import sys, os, subprocess, json, time, urllib.request, hashlib, shutil, re, signal, fcntl
# Turn off SDXL to run only the upscalers without its 7.1 GB checkpoint.
LOAD_SDXL = True
ENABLE_SEEDVR2 = True
# Optional gated model: accept its terms on Hugging Face and add Kaggle Secret HF_TOKEN.
ENABLE_FLUX3 = False
REQUIRE_ATTACHED_MODELS = True
# The builder embeds the shared asset catalog here.
from pathlib import Path
import hashlib
MODEL='Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors'
MODEL_SHA='c9e3e68f89b8e38689e1097d4be4573cf308de4e3fd044c64ca697bdb4aa8bca'
MODEL_SIZE=7105348188
def model_assets(load_sdxl=True, enable_seedvr2=True, enable_flux3=False):
 assets=[('upscale_models','RealESRGAN_x4plus.pth','https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth','4fa0d38905f75ac06eb49a7951b426670021be3018265fd191d2125df9d682f1',67040989)]
 if load_sdxl:assets.append(('checkpoints',MODEL,'https://huggingface.co/RunDiffusion/Juggernaut-XL-v9/resolve/main/'+MODEL,MODEL_SHA,MODEL_SIZE))
 if enable_seedvr2:
  prefix='https://huggingface.co/Comfy-Org/SeedVR2/resolve/df48879708206a403d2a61acd55578c2e80fd233/'
  assets.extend([
   ('diffusion_models','seedvr2_3b_int8_convrot.safetensors',prefix+'diffusion_models/seedvr2_3b_int8_convrot.safetensors','c3dec8bcc5916843a8a858572970597462e1f2dc598d6dfd818f6cd40f53a157',3458259704),
   ('vae','seedvr2_ema_vae_fp16.safetensors',prefix+'vae/seedvr2_ema_vae_fp16.safetensors','20678548f420d98d26f11442d3528f8b8c94e57ee046ef93dbb7633da8612ca1',501324814)])
 if enable_flux3:
  prefix='https://huggingface.co/black-forest-labs/flux-3-image-klein-comfy/resolve/9d36114cf7a2e03a45c28f5cc4987d20ebfdb6c1/'
  assets.extend([
   ('diffusion_models','flux3-image-klein-int8-convrot.safetensors',prefix+'diffusion_models/flux3-image-klein-int8-convrot.safetensors','847d8933548afd93df4af2cac2498e70891bf28d9dae06d695f536992e168fe2',5827874779),
   ('text_encoders','qwen3vl_4b_int8_convrot.safetensors',prefix+'text_encoders/qwen3vl_4b_int8_convrot.safetensors','32de113cbf8282b02cbff0794b637b9d6676da09314a1db5a30bcea9bf4ee01c',4858559240),
   ('vae','flux2-vae.safetensors',prefix+'vae/flux2-vae.safetensors','868fe7b343cc8f3a19dbcfcafbc3d5f888802be3f89bd81b65b3621a066ce8f3',336211292),
   ('text_encoders','qwen3.5_9b_flux3_prompt_upsampler_w4a8.safetensors',prefix+'text_encoders/qwen3.5_9b_flux3_prompt_upsampler_w4a8.safetensors','e5d85a7421d197bf5a12474511936ecabce19a833252f16e9310a983194a4458',6724090780)])
 return assets

WORKFLOW_PAYLOAD = {'Create': {'last_node_id': 7, 'last_link_id': 9, 'nodes': [{'id': 1, 'type': 'CheckpointLoaderSimple', 'pos': [0, 0], 'size': [320, 250], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [3]}, {'name': 'CLIP', 'type': 'CLIP', 'links': [1, 2]}, {'name': 'VAE', 'type': 'VAE', 'links': [8]}], 'properties': {'Node name for S&R': 'CheckpointLoaderSimple'}, 'widgets_values': ['Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors']}, {'id': 2, 'type': 'CLIPTextEncode', 'pos': [380, 0], 'size': [320, 250], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [4]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': ['Cinematic photograph of a lone explorer overlooking a vast alien coastline, dramatic clouds, natural light, detailed textures, thoughtful composition'], 'title': 'Describe your image'}, {'id': 3, 'type': 'CLIPTextEncode', 'pos': [380, 300], 'size': [320, 250], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 2}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [5]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': ['blurry, low quality, distorted anatomy, watermark, oversaturated'], 'title': 'Negative prompt'}, {'id': 4, 'type': 'EmptyLatentImage', 'pos': [380, 650], 'size': [320, 250], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [6]}], 'properties': {'Node name for S&R': 'EmptyLatentImage'}, 'widgets_values': [1024, 1024, 1]}, {'id': 5, 'type': 'KSampler', 'pos': [770, 0], 'size': [320, 250], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 3}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 4}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 5}, {'name': 'latent_image', 'type': 'LATENT', 'link': 6}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [7]}], 'properties': {'Node name for S&R': 'KSampler'}, 'widgets_values': [42, 'randomize', 30, 5.5, 'dpmpp_2m', 'karras', 1.0]}, {'id': 6, 'type': 'VAEDecodeTiled', 'pos': [1130, 350], 'size': [320, 250], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 7}, {'name': 'vae', 'type': 'VAE', 'link': 8}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [9]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 64, 64, 8]}, {'id': 7, 'type': 'SaveImage', 'pos': [1490, 350], 'size': [320, 250], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 9}], 'outputs': [], 'properties': {'Node name for S&R': 'SaveImage'}, 'widgets_values': ['ImageStudio/Create']}], 'links': [[1, 1, 1, 2, 0, 'CLIP'], [2, 1, 1, 3, 0, 'CLIP'], [3, 1, 0, 5, 0, 'MODEL'], [4, 2, 0, 5, 1, 'CONDITIONING'], [5, 3, 0, 5, 2, 'CONDITIONING'], [6, 4, 0, 5, 3, 'LATENT'], [7, 5, 0, 6, 0, 'LATENT'], [8, 1, 2, 6, 1, 'VAE'], [9, 6, 0, 7, 0, 'IMAGE']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.65, 'offset': [180, 180]}}, 'version': 0.4}, 'Restyle': {'last_node_id': 8, 'last_link_id': 11, 'nodes': [{'id': 1, 'type': 'CheckpointLoaderSimple', 'pos': [0, 0], 'size': [320, 250], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [5]}, {'name': 'CLIP', 'type': 'CLIP', 'links': [1, 2]}, {'name': 'VAE', 'type': 'VAE', 'links': [4, 10]}], 'properties': {'Node name for S&R': 'CheckpointLoaderSimple'}, 'widgets_values': ['Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors']}, {'id': 2, 'type': 'CLIPTextEncode', 'pos': [380, 0], 'size': [320, 250], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [6]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': ['Cinematic photograph of a lone explorer overlooking a vast alien coastline, dramatic clouds, natural light, detailed textures, thoughtful composition'], 'title': 'Describe your image'}, {'id': 3, 'type': 'CLIPTextEncode', 'pos': [380, 300], 'size': [320, 250], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 2}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [7]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': ['blurry, low quality, distorted anatomy, watermark, oversaturated'], 'title': 'Negative prompt'}, {'id': 4, 'type': 'LoadImage', 'pos': [0, 400], 'size': [320, 250], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [3]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'Upload a source image'}, {'id': 5, 'type': 'VAEEncode', 'pos': [380, 650], 'size': [320, 250], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'pixels', 'type': 'IMAGE', 'link': 3}, {'name': 'vae', 'type': 'VAE', 'link': 4}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [8]}], 'properties': {'Node name for S&R': 'VAEEncode'}, 'widgets_values': []}, {'id': 6, 'type': 'KSampler', 'pos': [770, 0], 'size': [320, 250], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 5}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 6}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 7}, {'name': 'latent_image', 'type': 'LATENT', 'link': 8}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [9]}], 'properties': {'Node name for S&R': 'KSampler'}, 'widgets_values': [42, 'randomize', 30, 5.5, 'dpmpp_2m', 'karras', 0.6]}, {'id': 7, 'type': 'VAEDecodeTiled', 'pos': [1130, 350], 'size': [320, 250], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 9}, {'name': 'vae', 'type': 'VAE', 'link': 10}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [11]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 64, 64, 8]}, {'id': 8, 'type': 'SaveImage', 'pos': [1490, 350], 'size': [320, 250], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 11}], 'outputs': [], 'properties': {'Node name for S&R': 'SaveImage'}, 'widgets_values': ['ImageStudio/Restyle']}], 'links': [[1, 1, 1, 2, 0, 'CLIP'], [2, 1, 1, 3, 0, 'CLIP'], [3, 4, 0, 5, 0, 'IMAGE'], [4, 1, 2, 5, 1, 'VAE'], [5, 1, 0, 6, 0, 'MODEL'], [6, 2, 0, 6, 1, 'CONDITIONING'], [7, 3, 0, 6, 2, 'CONDITIONING'], [8, 5, 0, 6, 3, 'LATENT'], [9, 6, 0, 7, 0, 'LATENT'], [10, 1, 2, 7, 1, 'VAE'], [11, 7, 0, 8, 0, 'IMAGE']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.65, 'offset': [180, 180]}}, 'version': 0.4}, 'Inpaint': {'last_node_id': 8, 'last_link_id': 12, 'nodes': [{'id': 1, 'type': 'CheckpointLoaderSimple', 'pos': [0, 0], 'size': [320, 250], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [6]}, {'name': 'CLIP', 'type': 'CLIP', 'links': [1, 2]}, {'name': 'VAE', 'type': 'VAE', 'links': [5, 11]}], 'properties': {'Node name for S&R': 'CheckpointLoaderSimple'}, 'widgets_values': ['Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors']}, {'id': 2, 'type': 'CLIPTextEncode', 'pos': [380, 0], 'size': [320, 250], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [7]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': ['Cinematic photograph of a lone explorer overlooking a vast alien coastline, dramatic clouds, natural light, detailed textures, thoughtful composition'], 'title': 'Describe your image'}, {'id': 3, 'type': 'CLIPTextEncode', 'pos': [380, 300], 'size': [320, 250], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 2}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [8]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': ['blurry, low quality, distorted anatomy, watermark, oversaturated'], 'title': 'Negative prompt'}, {'id': 4, 'type': 'LoadImage', 'pos': [0, 400], 'size': [320, 250], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [4]}, {'name': 'MASK', 'type': 'MASK', 'links': [3]}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'Upload image; right-click to open MaskEditor'}, {'id': 5, 'type': 'VAEEncodeForInpaint', 'pos': [380, 650], 'size': [320, 250], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'pixels', 'type': 'IMAGE', 'link': 4}, {'name': 'vae', 'type': 'VAE', 'link': 5}, {'name': 'mask', 'type': 'MASK', 'link': 3}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [9]}], 'properties': {'Node name for S&R': 'VAEEncodeForInpaint'}, 'widgets_values': [6]}, {'id': 6, 'type': 'KSampler', 'pos': [770, 0], 'size': [320, 250], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 6}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 7}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 8}, {'name': 'latent_image', 'type': 'LATENT', 'link': 9}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [10]}], 'properties': {'Node name for S&R': 'KSampler'}, 'widgets_values': [42, 'randomize', 30, 5.5, 'dpmpp_2m', 'karras', 1.0]}, {'id': 7, 'type': 'VAEDecodeTiled', 'pos': [1130, 350], 'size': [320, 250], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 10}, {'name': 'vae', 'type': 'VAE', 'link': 11}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [12]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 64, 64, 8]}, {'id': 8, 'type': 'SaveImage', 'pos': [1490, 350], 'size': [320, 250], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 12}], 'outputs': [], 'properties': {'Node name for S&R': 'SaveImage'}, 'widgets_values': ['ImageStudio/Inpaint']}], 'links': [[1, 1, 1, 2, 0, 'CLIP'], [2, 1, 1, 3, 0, 'CLIP'], [3, 4, 1, 5, 2, 'MASK'], [4, 4, 0, 5, 0, 'IMAGE'], [5, 1, 2, 5, 1, 'VAE'], [6, 1, 0, 6, 0, 'MODEL'], [7, 2, 0, 6, 1, 'CONDITIONING'], [8, 3, 0, 6, 2, 'CONDITIONING'], [9, 5, 0, 6, 3, 'LATENT'], [10, 6, 0, 7, 0, 'LATENT'], [11, 1, 2, 7, 1, 'VAE'], [12, 7, 0, 8, 0, 'IMAGE']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.65, 'offset': [180, 180]}}, 'version': 0.4}, 'Detail': {'last_node_id': 9, 'last_link_id': 14, 'nodes': [{'id': 1, 'type': 'CheckpointLoaderSimple', 'pos': [0, 0], 'size': [320, 250], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [3, 8]}, {'name': 'CLIP', 'type': 'CLIP', 'links': [1, 2]}, {'name': 'VAE', 'type': 'VAE', 'links': [13]}], 'properties': {'Node name for S&R': 'CheckpointLoaderSimple'}, 'widgets_values': ['Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors']}, {'id': 2, 'type': 'CLIPTextEncode', 'pos': [380, 0], 'size': [320, 250], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [4, 9]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': ['Cinematic photograph of a lone explorer overlooking a vast alien coastline, dramatic clouds, natural light, detailed textures, thoughtful composition'], 'title': 'Describe your image'}, {'id': 3, 'type': 'CLIPTextEncode', 'pos': [380, 300], 'size': [320, 250], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 2}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [5, 10]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': ['blurry, low quality, distorted anatomy, watermark, oversaturated'], 'title': 'Negative prompt'}, {'id': 4, 'type': 'EmptyLatentImage', 'pos': [380, 650], 'size': [320, 250], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [6]}], 'properties': {'Node name for S&R': 'EmptyLatentImage'}, 'widgets_values': [1024, 1024, 1]}, {'id': 5, 'type': 'KSampler', 'pos': [770, 0], 'size': [320, 250], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 3}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 4}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 5}, {'name': 'latent_image', 'type': 'LATENT', 'link': 6}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [7]}], 'properties': {'Node name for S&R': 'KSampler'}, 'widgets_values': [42, 'randomize', 30, 5.5, 'dpmpp_2m', 'karras', 1.0]}, {'id': 6, 'type': 'LatentUpscaleBy', 'pos': [1130, 0], 'size': [320, 250], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 7}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [11]}], 'properties': {'Node name for S&R': 'LatentUpscaleBy'}, 'widgets_values': ['bislerp', 1.5]}, {'id': 7, 'type': 'KSampler', 'pos': [1490, 0], 'size': [320, 250], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 8}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 9}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 10}, {'name': 'latent_image', 'type': 'LATENT', 'link': 11}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [12]}], 'properties': {'Node name for S&R': 'KSampler'}, 'widgets_values': [42, 'fixed', 20, 5.0, 'dpmpp_2m', 'karras', 0.3], 'title': 'Detail pass — 1536px; costs more VRAM'}, {'id': 8, 'type': 'VAEDecodeTiled', 'pos': [1900, 350], 'size': [320, 250], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 12}, {'name': 'vae', 'type': 'VAE', 'link': 13}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [14]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 64, 64, 8]}, {'id': 9, 'type': 'SaveImage', 'pos': [2260, 350], 'size': [320, 250], 'flags': {}, 'order': 8, 'mode': 0, 'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 14}], 'outputs': [], 'properties': {'Node name for S&R': 'SaveImage'}, 'widgets_values': ['ImageStudio/Detail']}], 'links': [[1, 1, 1, 2, 0, 'CLIP'], [2, 1, 1, 3, 0, 'CLIP'], [3, 1, 0, 5, 0, 'MODEL'], [4, 2, 0, 5, 1, 'CONDITIONING'], [5, 3, 0, 5, 2, 'CONDITIONING'], [6, 4, 0, 5, 3, 'LATENT'], [7, 5, 0, 6, 0, 'LATENT'], [8, 1, 0, 7, 0, 'MODEL'], [9, 2, 0, 7, 1, 'CONDITIONING'], [10, 3, 0, 7, 2, 'CONDITIONING'], [11, 6, 0, 7, 3, 'LATENT'], [12, 7, 0, 8, 0, 'LATENT'], [13, 1, 2, 8, 1, 'VAE'], [14, 8, 0, 9, 0, 'IMAGE']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.65, 'offset': [180, 180]}}, 'version': 0.4}, 'Upscale 2x': {'last_node_id': 5, 'last_link_id': 4, 'nodes': [{'id': 1, 'type': 'LoadImage', 'pos': [0, 100], 'size': [320, 250], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [2]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'Upload an image to enlarge'}, {'id': 2, 'type': 'UpscaleModelLoader', 'pos': [0, 440], 'size': [320, 250], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'UPSCALE_MODEL', 'type': 'UPSCALE_MODEL', 'links': [1]}], 'properties': {'Node name for S&R': 'UpscaleModelLoader'}, 'widgets_values': ['RealESRGAN_x4plus.pth']}, {'id': 3, 'type': 'ImageUpscaleWithModel', 'pos': [360, 100], 'size': [320, 250], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [{'name': 'upscale_model', 'type': 'UPSCALE_MODEL', 'link': 1}, {'name': 'image', 'type': 'IMAGE', 'link': 2}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [3]}], 'properties': {'Node name for S&R': 'ImageUpscaleWithModel'}, 'widgets_values': []}, {'id': 4, 'type': 'ImageScaleBy', 'pos': [720, 100], 'size': [320, 250], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 3}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [4]}], 'properties': {'Node name for S&R': 'ImageScaleBy'}, 'widgets_values': ['lanczos', 0.5], 'title': '4x model then downsample to 2x'}, {'id': 5, 'type': 'SaveImage', 'pos': [1080, 100], 'size': [320, 250], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 4}], 'outputs': [], 'properties': {'Node name for S&R': 'SaveImage'}, 'widgets_values': ['ImageStudio/Upscale 2x']}], 'links': [[1, 2, 0, 3, 0, 'UPSCALE_MODEL'], [2, 1, 0, 3, 1, 'IMAGE'], [3, 3, 0, 4, 0, 'IMAGE'], [4, 4, 0, 5, 0, 'IMAGE']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.65, 'offset': [180, 180]}}, 'version': 0.4}, 'Upscale 4x': {'last_node_id': 4, 'last_link_id': 3, 'nodes': [{'id': 1, 'type': 'LoadImage', 'pos': [0, 100], 'size': [320, 250], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [2]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'Upload an image to enlarge'}, {'id': 2, 'type': 'UpscaleModelLoader', 'pos': [0, 440], 'size': [320, 250], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'UPSCALE_MODEL', 'type': 'UPSCALE_MODEL', 'links': [1]}], 'properties': {'Node name for S&R': 'UpscaleModelLoader'}, 'widgets_values': ['RealESRGAN_x4plus.pth']}, {'id': 3, 'type': 'ImageUpscaleWithModel', 'pos': [360, 100], 'size': [320, 250], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [{'name': 'upscale_model', 'type': 'UPSCALE_MODEL', 'link': 1}, {'name': 'image', 'type': 'IMAGE', 'link': 2}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [3]}], 'properties': {'Node name for S&R': 'ImageUpscaleWithModel'}, 'widgets_values': []}, {'id': 4, 'type': 'SaveImage', 'pos': [1080, 100], 'size': [320, 250], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 3}], 'outputs': [], 'properties': {'Node name for S&R': 'SaveImage'}, 'widgets_values': ['ImageStudio/Upscale 4x']}], 'links': [[1, 2, 0, 3, 0, 'UPSCALE_MODEL'], [2, 1, 0, 3, 1, 'IMAGE'], [3, 3, 0, 4, 0, 'IMAGE']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.65, 'offset': [180, 180]}}, 'version': 0.4}, 'Upscale Creative': {'last_node_id': 7, 'last_link_id': 9, 'nodes': [{'id': 1, 'type': 'CheckpointLoaderSimple', 'pos': [0, 0], 'size': [320, 250], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [4]}, {'name': 'CLIP', 'type': 'CLIP', 'links': [1, 2]}, {'name': 'VAE', 'type': 'VAE', 'links': [7]}], 'properties': {'Node name for S&R': 'CheckpointLoaderSimple'}, 'widgets_values': ['Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors']}, {'id': 2, 'type': 'CLIPTextEncode', 'pos': [380, 0], 'size': [320, 250], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [5]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': ['Cinematic photograph of a lone explorer overlooking a vast alien coastline, dramatic clouds, natural light, detailed textures, thoughtful composition'], 'title': 'Describe your image'}, {'id': 3, 'type': 'CLIPTextEncode', 'pos': [380, 300], 'size': [320, 250], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 2}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [6]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': ['blurry, low quality, distorted anatomy, watermark, oversaturated'], 'title': 'Negative prompt'}, {'id': 4, 'type': 'LoadImage', 'pos': [0, 400], 'size': [320, 250], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [3]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'Upload source; 2x creative enhancement'}, {'id': 5, 'type': 'UpscaleModelLoader', 'pos': [380, 650], 'size': [320, 250], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'UPSCALE_MODEL', 'type': 'UPSCALE_MODEL', 'links': [8]}], 'properties': {'Node name for S&R': 'UpscaleModelLoader'}, 'widgets_values': ['RealESRGAN_x4plus.pth']}, {'id': 6, 'type': 'UltimateSDUpscale', 'pos': [800, 0], 'size': [320, 250], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 3}, {'name': 'model', 'type': 'MODEL', 'link': 4}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 5}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 6}, {'name': 'vae', 'type': 'VAE', 'link': 7}, {'name': 'upscale_model', 'type': 'UPSCALE_MODEL', 'link': 8}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [9]}], 'properties': {'Node name for S&R': 'UltimateSDUpscale'}, 'widgets_values': [2.0, 42, 'fixed', 20, 5.0, 'dpmpp_2m', 'karras', 0.2, 'Linear', 768, 768, 8, 64, 'Half Tile', 0.15, 64, 8, 32, True, True, 1], 'title': 'Creative 2x — tiles can change faces and lettering'}, {'id': 7, 'type': 'SaveImage', 'pos': [1200, 0], 'size': [320, 250], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 9}], 'outputs': [], 'properties': {'Node name for S&R': 'SaveImage'}, 'widgets_values': ['ImageStudio/Creative']}], 'links': [[1, 1, 1, 2, 0, 'CLIP'], [2, 1, 1, 3, 0, 'CLIP'], [3, 4, 0, 6, 0, 'IMAGE'], [4, 1, 0, 6, 1, 'MODEL'], [5, 2, 0, 6, 2, 'CONDITIONING'], [6, 3, 0, 6, 3, 'CONDITIONING'], [7, 1, 2, 6, 4, 'VAE'], [8, 5, 0, 6, 5, 'UPSCALE_MODEL'], [9, 6, 0, 7, 0, 'IMAGE']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.65, 'offset': [180, 180]}}, 'version': 0.4}, 'Restore SeedVR2': {'last_node_id': 11, 'last_link_id': 15, 'nodes': [{'id': 1, 'type': 'LoadImage', 'pos': [0, 100], 'size': [320, 250], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [1]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'Upload an image to enlarge'}, {'id': 2, 'type': 'ImageScaleBy', 'pos': [360, 100], 'size': [320, 250], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 1}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [2, 14]}], 'properties': {'Node name for S&R': 'ImageScaleBy'}, 'widgets_values': ['lanczos', 2.0], 'title': '2x target; start with source at 1024px or smaller'}, {'id': 3, 'type': 'SeedVR2Preprocess', 'pos': [720, 100], 'size': [320, 250], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [{'name': 'resized_images', 'type': 'IMAGE', 'link': 2}], 'outputs': [{'name': 'images', 'type': 'IMAGE', 'links': [3]}], 'properties': {'Node name for S&R': 'SeedVR2Preprocess'}, 'widgets_values': []}, {'id': 4, 'type': 'VAELoader', 'pos': [360, 440], 'size': [320, 250], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'VAE', 'type': 'VAE', 'links': [4, 12]}], 'properties': {'Node name for S&R': 'VAELoader'}, 'widgets_values': ['seedvr2_ema_vae_fp16.safetensors']}, {'id': 5, 'type': 'UNETLoader', 'pos': [0, 440], 'size': [320, 250], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [5, 7]}], 'properties': {'Node name for S&R': 'UNETLoader'}, 'widgets_values': ['seedvr2_3b_int8_convrot.safetensors', 'default']}, {'id': 6, 'type': 'VAEEncodeTiled', 'pos': [1080, 100], 'size': [320, 250], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [{'name': 'pixels', 'type': 'IMAGE', 'link': 3}, {'name': 'vae', 'type': 'VAE', 'link': 4}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [6, 10]}], 'properties': {'Node name for S&R': 'VAEEncodeTiled'}, 'widgets_values': [512, 128, 64, 8]}, {'id': 7, 'type': 'SeedVR2Conditioning', 'pos': [1080, 440], 'size': [320, 250], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 5}, {'name': 'vae_conditioning', 'type': 'LATENT', 'link': 6}], 'outputs': [{'name': 'positive', 'type': 'CONDITIONING', 'links': [8]}, {'name': 'negative', 'type': 'CONDITIONING', 'links': [9]}], 'properties': {'Node name for S&R': 'SeedVR2Conditioning'}, 'widgets_values': []}, {'id': 8, 'type': 'KSampler', 'pos': [1440, 100], 'size': [320, 250], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 7}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 8}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 9}, {'name': 'latent_image', 'type': 'LATENT', 'link': 10}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [11]}], 'properties': {'Node name for S&R': 'KSampler'}, 'widgets_values': [42, 'fixed', 1, 1.0, 'euler', 'simple', 1.0]}, {'id': 9, 'type': 'VAEDecodeTiled', 'pos': [1800, 100], 'size': [320, 250], 'flags': {}, 'order': 8, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 11}, {'name': 'vae', 'type': 'VAE', 'link': 12}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [13]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 128, 64, 8]}, {'id': 10, 'type': 'SeedVR2PostProcessing', 'pos': [2160, 100], 'size': [320, 250], 'flags': {}, 'order': 9, 'mode': 0, 'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 13}, {'name': 'original_resized_images', 'type': 'IMAGE', 'link': 14}], 'outputs': [{'name': 'images', 'type': 'IMAGE', 'links': [15]}], 'properties': {'Node name for S&R': 'SeedVR2PostProcessing'}, 'widgets_values': ['none']}, {'id': 11, 'type': 'SaveImage', 'pos': [2520, 100], 'size': [320, 250], 'flags': {}, 'order': 10, 'mode': 0, 'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 15}], 'outputs': [], 'properties': {'Node name for S&R': 'SaveImage'}, 'widgets_values': ['ImageStudio/Restore SeedVR2']}], 'links': [[1, 1, 0, 2, 0, 'IMAGE'], [2, 2, 0, 3, 0, 'IMAGE'], [3, 3, 0, 6, 0, 'IMAGE'], [4, 4, 0, 6, 1, 'VAE'], [5, 5, 0, 7, 0, 'MODEL'], [6, 6, 0, 7, 1, 'LATENT'], [7, 5, 0, 8, 0, 'MODEL'], [8, 7, 0, 8, 1, 'CONDITIONING'], [9, 7, 1, 8, 2, 'CONDITIONING'], [10, 6, 0, 8, 3, 'LATENT'], [11, 8, 0, 9, 0, 'LATENT'], [12, 4, 0, 9, 1, 'VAE'], [13, 9, 0, 10, 0, 'IMAGE'], [14, 2, 0, 10, 1, 'IMAGE'], [15, 10, 0, 11, 0, 'IMAGE']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.65, 'offset': [180, 180]}}, 'version': 0.4}, 'Create FLUX3': {'last_node_id': 30, 'last_link_id': 91, 'nodes': [{'id': 17, 'type': 'PrimitiveStringMultiline', 'pos': [-680, -950], 'size': [490, 450], 'flags': {}, 'order': 13, 'mode': 0, 'inputs': [], 'outputs': [{'localized_name': 'STRING', 'name': 'STRING', 'type': 'STRING', 'links': [17]}], 'properties': {'Node name for S&R': 'PrimitiveStringMultiline'}, 'widgets_values': ['A cinematic photograph of a lighthouse above a stormy coastline at sunset.']}, {'id': 12, 'type': 'CLIPLoader', 'pos': [-140, -930], 'size': [500, 110], 'flags': {}, 'order': 10, 'mode': 0, 'inputs': [], 'outputs': [{'localized_name': 'CLIP', 'name': 'CLIP', 'type': 'CLIP', 'links': [10]}], 'properties': {'Node name for S&R': 'CLIPLoader', 'models': [{'name': 'qwen3.5_9b_flux3_prompt_upsampler_w4a8.safetensors', 'url': 'https://huggingface.co/black-forest-labs/flux-3-image-klein-comfy/resolve/main/text_encoders/qwen3.5_9b_flux3_prompt_upsampler_w4a8.safetensors', 'directory': 'text_encoders'}]}, 'widgets_values': ['qwen3.5_9b_flux3_prompt_upsampler_w4a8.safetensors', 'flux3', 'default']}, {'id': 14, 'type': 'PrimitiveStringMultiline', 'pos': [-150, -750], 'size': [530, 700], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'localized_name': 'STRING', 'name': 'STRING', 'type': 'STRING', 'links': [13]}], 'title': 'Text (System prompt)', 'properties': {'Node name for S&R': 'PrimitiveStringMultiline'}, 'widgets_values': ['You are the prompt upsampler of a text-to-image model. Expand the user\'s short request into ONE detailed, self-contained caption: the exact text the image model receives. Output only the caption, with no greetings, explanations, refusals, questions, Markdown, headings, JSON, coordinates, ids or angle-bracket tokens, and no quotes around it. If the input is not clearly an image request, caption a visualization of its concept. The request may begin with a "TARGET CANVAS: W:H" line: compose for that canvas (wide spreads content sideways, tall stacks it) and never mention the ratio or the canvas.\n\nFidelity and planning: keep every explicit subject, attribute, count, spatial constraint, style cue and piece of wording; never contradict, substitute, drop or narrow the request, and keep an abstract concept abstract. Decide everything left open as ONE concrete, consistent plan; never offer alternatives or hedge. Never re-express a stated constraint through another mechanism; add camera optics (depth of field, bokeh, blur) only when asked for or implied. Without a stated style, plan an actual photograph and open with the medium convention ("Full-color outdoor photograph, ...", "Full-color indoor photograph, ...", "Studio photograph, ...", "Close-up photograph, ..."); keep "Photorealistic" for renders, and turn "photorealistic" or "realistic" requests for photos into the photograph convention. Give a photograph a few ordinary, non-ideal specifics drawn from the scene\'s own objects instead of the stock-photo ideal. Do not invent names, brands, logos or celebrities; keep user-named ones. A long request keeps its content but not its wording: drop appraisal and sales vocabulary, turn moods into observable physical facts, and add only new concrete content with positions. Time of day is lighting, not rendered text.\n\nCaption format: one continuous prose block in the generative voice, never "this image shows", "we can see" or "is visible". Lead with medium and style, then scene, lighting and camera, then the subjects with concrete colors, materials and state. Every person in focus gets an observable profile: ethnicity, gender, age range, skin tone ("fair skin", "brown skin"), hair and clothing. Make the layout reconstructable from words: frame-relative positions (frame-left, frame-right, upper, lower, foreground, midground, background), sentence-initial anchors ("On the left, ...", "In the background, ..."), relations (holds, faces, behind) and a consistent order; describe up to about ten instances one by one, beyond that give the exact count and arrangement. Out-of-focus or distant planes are shapes, colors and silhouettes, never identities or readable text, and the background takes about a sixth of the caption. Be definite: no hedging adverbs, no quality or emotion judgments, no closing sentence about mood; state negations as what is present. Rendered text: transcribe every wording verbatim inside double quotes in its original language and script, keep the line breaks of multi-line text, and give its typography, color and placement; author concrete, domain-correct wording when the request needs text it does not supply, and name canonical structures (calendar, keyboard, periodic table) exactly with their full layout. All prose is English; only quoted text keeps its language. Typical length is 100 to 350 words, more for complex scenes.\n\nOutput only the caption.\n']}, {'id': 11, 'type': 'TextGenerate', 'pos': [400, -930], 'size': [400, 460], 'flags': {}, 'order': 9, 'mode': 0, 'inputs': [{'localized_name': 'clip', 'name': 'clip', 'type': 'CLIP', 'link': 10}, {'localized_name': 'image', 'name': 'image', 'shape': 7, 'type': 'IMAGE', 'link': None}, {'localized_name': 'video', 'name': 'video', 'shape': 7, 'type': 'IMAGE', 'link': None}, {'localized_name': 'audio', 'name': 'audio', 'shape': 7, 'type': 'AUDIO', 'link': None}, {'localized_name': 'system_prompt', 'name': 'system_prompt', 'shape': 7, 'type': 'STRING', 'link': 13}, {'localized_name': 'prompt', 'name': 'prompt', 'type': 'STRING', 'widget': {'name': 'prompt'}, 'link': 17}], 'outputs': [{'localized_name': 'generated_text', 'name': 'generated_text', 'type': 'STRING', 'links': [90]}, {'localized_name': 'thinking', 'name': 'thinking', 'type': 'STRING', 'links': []}], 'properties': {'Node name for S&R': 'TextGenerate'}, 'widgets_values': ['', 3584, 'on', 1, 1, 0.95, 0, 1.05, 0, 1.5, False, True, 'off']}, {'id': 1, 'type': 'UNETLoader', 'pos': [-150, 60], 'size': [340, 90], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'localized_name': 'MODEL', 'name': 'MODEL', 'type': 'MODEL', 'slot_index': 0, 'links': [1]}], 'properties': {'Node name for S&R': 'UNETLoader', 'models': [{'name': 'flux3-image-klein-int8-convrot.safetensors', 'url': 'https://huggingface.co/black-forest-labs/flux-3-image-klein-comfy/resolve/main/diffusion_models/flux3-image-klein-int8-convrot.safetensors', 'directory': 'diffusion_models'}]}, 'widgets_values': ['flux3-image-klein-int8-convrot.safetensors', 'default'], 'color': '#223', 'bgcolor': '#335'}, {'id': 4, 'type': 'CLIPTextEncode', 'pos': [250, 90], 'size': [450, 460], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'localized_name': 'clip', 'name': 'clip', 'type': 'CLIP', 'link': 2}, {'localized_name': 'text', 'name': 'text', 'type': 'STRING', 'widget': {'name': 'text'}, 'link': 20}], 'outputs': [{'localized_name': 'CONDITIONING', 'name': 'CONDITIONING', 'type': 'CONDITIONING', 'slot_index': 0, 'links': [8]}], 'title': 'Positive Prompt', 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': ['A close-up photograph of a red fox standing in fresh snow at sunrise. The fox faces the camera with alert ears and amber eyes, its thick orange fur dusted with snowflakes. Soft golden light comes from the left, the background is a blurred line of snow-covered pine trees, shallow depth of field.'], 'color': '#232', 'bgcolor': '#353'}, {'id': 2, 'type': 'CLIPLoader', 'pos': [-150, 220], 'size': [340, 110], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [], 'outputs': [{'localized_name': 'CLIP', 'name': 'CLIP', 'type': 'CLIP', 'slot_index': 0, 'links': [2, 3]}], 'properties': {'Node name for S&R': 'CLIPLoader', 'models': [{'name': 'qwen3vl_4b_int8_convrot.safetensors', 'url': 'https://huggingface.co/black-forest-labs/flux-3-image-klein-comfy/resolve/main/text_encoders/qwen3vl_4b_int8_convrot.safetensors', 'directory': 'text_encoders'}]}, 'widgets_values': ['qwen3vl_4b_int8_convrot.safetensors', 'flux3', 'default'], 'color': '#432', 'bgcolor': '#653'}, {'id': 7, 'type': 'KSampler', 'pos': [760, 30], 'size': [320, 480], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [{'localized_name': 'model', 'name': 'model', 'type': 'MODEL', 'link': 1}, {'localized_name': 'positive', 'name': 'positive', 'type': 'CONDITIONING', 'link': 8}, {'localized_name': 'negative', 'name': 'negative', 'type': 'CONDITIONING', 'link': 9}, {'localized_name': 'latent_image', 'name': 'latent_image', 'type': 'LATENT', 'link': 39}], 'outputs': [{'localized_name': 'LATENT', 'name': 'LATENT', 'type': 'LATENT', 'slot_index': 0, 'links': [4]}], 'properties': {'Node name for S&R': 'KSampler'}, 'widgets_values': [42, 'randomize', 16, 1.0, 'euler', 'simple', 1.0]}, {'id': 3, 'type': 'VAELoader', 'pos': [-150, 400], 'size': [340, 60], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [], 'outputs': [{'localized_name': 'VAE', 'name': 'VAE', 'type': 'VAE', 'slot_index': 0, 'links': [5]}], 'properties': {'Node name for S&R': 'VAELoader', 'models': [{'name': 'flux2-vae.safetensors', 'url': 'https://huggingface.co/black-forest-labs/flux-3-image-klein-comfy/resolve/main/vae/flux2-vae.safetensors', 'directory': 'vae'}]}, 'widgets_values': ['flux2-vae.safetensors'], 'color': '#323', 'bgcolor': '#535'}, {'id': 13, 'type': 'PreviewAny', 'pos': [830, -660], 'size': [350, 490], 'flags': {}, 'order': 11, 'mode': 0, 'inputs': [{'localized_name': 'source', 'name': 'source', 'type': '*', 'link': 90}], 'outputs': [{'localized_name': 'STRING', 'name': 'STRING', 'type': 'STRING', 'links': [20]}], 'properties': {'Node name for S&R': 'PreviewAny'}, 'widgets_values': []}, {'id': 6, 'type': 'EmptyFlux2LatentImage', 'pos': [-120, 590], 'size': [300, 110], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [], 'outputs': [{'localized_name': 'LATENT', 'name': 'LATENT', 'type': 'LATENT', 'slot_index': 0, 'links': [39]}], 'properties': {'Node name for S&R': 'EmptyFlux2LatentImage'}, 'widgets_values': [1024, 1024, 1], 'color': '#323', 'bgcolor': '#535'}, {'id': 5, 'type': 'CLIPTextEncode', 'pos': [250, 610], 'size': [460, 110], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [{'localized_name': 'clip', 'name': 'clip', 'type': 'CLIP', 'link': 3}], 'outputs': [{'localized_name': 'CONDITIONING', 'name': 'CONDITIONING', 'type': 'CONDITIONING', 'slot_index': 0, 'links': [9]}], 'title': 'Negative Prompt (empty = reference default)', 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': [''], 'color': '#223', 'bgcolor': '#335'}, {'id': 8, 'type': 'VAEDecodeTiled', 'pos': [1120, 30], 'size': [210, 50], 'flags': {}, 'order': 8, 'mode': 0, 'inputs': [{'localized_name': 'samples', 'name': 'samples', 'type': 'LATENT', 'link': 4}, {'localized_name': 'vae', 'name': 'vae', 'type': 'VAE', 'link': 5}], 'outputs': [{'localized_name': 'IMAGE', 'name': 'IMAGE', 'type': 'IMAGE', 'slot_index': 0, 'links': [91]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 64, 64, 8]}, {'id': 30, 'type': 'SaveImage', 'pos': [1400, 30], 'size': [320, 250], 'flags': {}, 'order': 30, 'mode': 0, 'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 91}], 'outputs': [], 'properties': {'Node name for S&R': 'SaveImage'}, 'widgets_values': ['ImageStudio/FLUX3']}], 'links': [[2, 2, 0, 4, 0, 'CLIP'], [20, 13, 0, 4, 1, 'STRING'], [1, 1, 0, 7, 0, 'MODEL'], [8, 4, 0, 7, 1, 'CONDITIONING'], [9, 5, 0, 7, 2, 'CONDITIONING'], [3, 2, 0, 5, 0, 'CLIP'], [4, 7, 0, 8, 0, 'LATENT'], [5, 3, 0, 8, 1, 'VAE'], [39, 6, 0, 7, 3, 'LATENT'], [10, 12, 0, 11, 0, 'CLIP'], [13, 14, 0, 11, 4, 'STRING'], [17, 17, 0, 11, 5, 'STRING'], [90, 11, 0, 13, 0, 'STRING'], [91, 8, 0, 30, 0, 'IMAGE']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.5, 'offset': [750, 1000]}}, 'version': 0.4}}
COMFY_REV='1332fabfcc507aa7c548cd5152071ed494c207e8'
MODEL='Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors'
MODEL_SHA='c9e3e68f89b8e38689e1097d4be4573cf308de4e3fd044c64ca697bdb4aa8bca'
MODEL_SIZE=7105348188
CF_SHA='1f99fb518b83dec2ba8d870a7cc991d82a77e6196295a3d2754ba17ab9f91872'
BASE=Path('/kaggle/temp/image-studio-v3'); BASE.mkdir(parents=True,exist_ok=True)
OUT=Path('/kaggle/working/image-studio'); OUT.mkdir(parents=True,exist_ok=True)
lock=open(BASE/'startup.lock','w')
try: fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)
except BlockingIOError: raise RuntimeError('Image Studio is already running. Stop its startup cell before starting again.')
children=[]
# Fail before downloading if the previous studio still owns this port.
import socket
probe=socket.socket()
try: probe.bind(('127.0.0.1',8188))
except OSError: raise RuntimeError('Port 8188 is occupied. Stop the earlier studio cell before starting v3.')
finally: probe.close()

def run(args,**kw):subprocess.run(list(map(str,args)),check=True,**kw)
def sha(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for chunk in iter(lambda:f.read(8*1024*1024),b''):h.update(chunk)
 return h.hexdigest()
def download(url,path,expected,size=None):
 if path.exists() and sha(path)==expected:return
 if path.exists():raise RuntimeError(f'Checksum mismatch: {path}. Preserve or move this file before retrying.')
 if size and shutil.disk_usage(path.parent).free<size+2*1024**3:raise RuntimeError('Insufficient free disk for model plus 2 GiB reserve.')
 temp=path.with_suffix(path.suffix+'.partial')
 print('Downloading',path.name,flush=True)
 with urllib.request.urlopen(url,timeout=120) as response,open(temp,'wb') as target:shutil.copyfileobj(response,target,8*1024*1024)
 if sha(temp)!=expected:raise RuntimeError(f'Download checksum mismatch: {temp}')
 temp.replace(path)
def spawn(args,log):
 f=open(log,'w');p=subprocess.Popen(list(map(str,args)),stdout=f,stderr=subprocess.STDOUT,start_new_session=True);f.close();children.append(p);return p
# Check all required names before package setup or GPU initialization.
if REQUIRE_ATTACHED_MODELS:
 missing=[]
 for folder,name,_,_,size in model_assets(LOAD_SDXL,ENABLE_SEEDVR2,ENABLE_FLUX3):
  candidates=list(Path('/kaggle/input').rglob(name)) if Path('/kaggle/input').exists() else []
  candidates.extend(Path('/kaggle/temp')/v/'ComfyUI/models'/folder/name for v in ['image-studio-v1','image-studio-v2','image-studio-v3'])
  if not any(p.is_file() and p.stat().st_size==size for p in candidates):missing.append(name)
 if missing:raise RuntimeError('Attach your saved Image Studio Model Packs output before starting. Missing: '+', '.join(missing)+'. No model downloads started.')
try:
 import torch
 if not torch.cuda.is_available():raise RuntimeError('Select a Kaggle GPU accelerator and restart this cell.')
 print('GPU:',torch.cuda.get_device_name(0),'Torch:',torch.__version__,flush=True)
 free,total=torch.cuda.mem_get_info(0)
 if free<10*1024**3:raise RuntimeError('GPU 0 has less than 10 GiB free. Stop other GPU notebooks/processes first.')
 # Separate T4s do not form one shared VRAM pool. This studio uses GPU 0.
 comfy=BASE/'ComfyUI'
 if not comfy.exists():
  run(['git','clone','--no-checkout','https://github.com/comfyanonymous/ComfyUI.git',comfy])
  run(['git','-C',comfy,'checkout','--detach',COMFY_REV])
 if subprocess.check_output(['git','-C',str(comfy),'rev-parse','HEAD'],text=True).strip()!=COMFY_REV:raise RuntimeError('Unexpected ComfyUI revision.')
 venv=BASE/'venv';python=venv/'bin/python'
 if not python.exists():run([sys.executable,'-m','venv','--system-site-packages','--without-pip',venv])
 constraints=BASE/'constraints.txt'
 import importlib.metadata as meta
 protected=['torch','torchvision','torchaudio','numpy','scipy']
 constraints.write_text('\n'.join(f'{p}=={meta.version(p)}' for p in protected))
 requirements=BASE/'requirements.txt'
 requirements.write_text('\n'.join(x for x in (comfy/'requirements.txt').read_text().splitlines() if x.strip() not in protected)+'\ntransformers==5.22.2\nhuggingface-hub==2.4.0\n')
 fingerprint=hashlib.sha256((requirements.read_text()+constraints.read_text()+sys.version).encode()).hexdigest()
 marker=BASE/'setup.complete'
 if not marker.exists() or marker.read_text()!=fingerprint:
  run([sys.executable,'-m','pip','--python',python,'install','-r',requirements,'-c',constraints])
  run([python,'-c','import torch, torchvision, transformers, aiohttp, safetensors; assert torch.cuda.is_available()'])
  marker.write_text(fingerprint)
 run([sys.executable,'-m','pip','--python',python,'freeze'],stdout=open(OUT/'packages.txt','w'))
 ultimate=comfy/'custom_nodes/ComfyUI_UltimateSDUpscale'
 if LOAD_SDXL:
  if not ultimate.exists():
   run(['git','clone','--no-checkout','https://github.com/ssitu/ComfyUI_UltimateSDUpscale.git',ultimate])
   run(['git','-C',ultimate,'checkout','--detach','a5547db9e1d07d3318bb21e9e9c474f4c1e9c8df'])
  if subprocess.check_output(['git','-C',str(ultimate),'rev-parse','HEAD'],text=True).strip()!='a5547db9e1d07d3318bb21e9e9c474f4c1e9c8df':raise RuntimeError('Unexpected Ultimate SD Upscale revision.')
  run(['git','-C',ultimate,'submodule','update','--init','--recursive'])
 assets=model_assets(LOAD_SDXL,ENABLE_SEEDVR2,ENABLE_FLUX3)
 for folder,filename,url,digest,size in assets:
  dest=comfy/'models'/folder/filename;dest.parent.mkdir(parents=True,exist_ok=True)
  verified=False
  if not dest.exists():
   candidates=list(Path('/kaggle/input').rglob(filename)) if Path('/kaggle/input').exists() else []
   candidates.extend(Path('/kaggle/temp')/v/'ComfyUI/models'/folder/filename for v in ['image-studio-v1','image-studio-v2'])
   for candidate in candidates:
    if candidate.is_file() and candidate.stat().st_size==size and sha(candidate)==digest:
     dest.symlink_to(candidate);verified=True;print('Reusing verified model:',candidate,flush=True);break
  if REQUIRE_ATTACHED_MODELS and not dest.exists():
   raise RuntimeError('Missing saved model: '+filename+'. Attach your saved Image Studio Model Packs output via Add Input. No model was downloaded.')
  if ENABLE_FLUX3 and 'black-forest-labs' in url and not dest.exists():
   if shutil.disk_usage(dest.parent).free<size+2*1024**3:raise RuntimeError('Insufficient disk space for selected FLUX3 assets.')
   from kaggle_secrets import UserSecretsClient
   try: token=UserSecretsClient().get_secret('HF_TOKEN')
   except Exception: raise RuntimeError('Enable Kaggle Secret HF_TOKEN with access to the accepted FLUX 3 model terms.') from None
   env=os.environ.copy();env['HF_TOKEN']=token
   try:
    code="import os;from huggingface_hub import hf_hub_download;hf_hub_download(repo_id='black-forest-labs/flux-3-image-klein-comfy',filename="+repr(folder+'/'+filename)+",revision='9d36114cf7a2e03a45c28f5cc4987d20ebfdb6c1',local_dir="+repr(str(comfy/'models'))+",token=os.environ['HF_TOKEN'])"
    run([python,'-c',code],env=env)
   finally:env.pop('HF_TOKEN',None);del token
  if not verified:download(url,dest,digest,size)
 # Only install workflows for enabled model families.
 WORKFLOW_PAYLOAD={k:v for k,v in WORKFLOW_PAYLOAD.items() if (LOAD_SDXL or k in ['Upscale 2x','Upscale 4x','Restore SeedVR2','Create FLUX3']) and (ENABLE_FLUX3 or k!='Create FLUX3') and (ENABLE_SEEDVR2 or k!='Restore SeedVR2')}
 workflow_dir=comfy/'user/default/workflows';workflow_dir.mkdir(parents=True,exist_ok=True)
 for name,wf in WORKFLOW_PAYLOAD.items():(workflow_dir/f'Image Studio - {name}.json').write_text(json.dumps(wf,indent=2))
 # Stable output destination: PNGs retain the actual graph, prompt, seed and settings.
 (OUT/'provenance.json').write_text(json.dumps({'comfy_revision':COMFY_REV,'model':MODEL,'sha256':MODEL_SHA,'assets':[{'file':f,'sha256':d,'bytes':z} for _,f,_,d,z in assets],'torch':torch.__version__},indent=2))
 cf=BASE/'cloudflared'
 download('https://github.com/cloudflare/cloudflared/releases/download/2026.10.7/cloudflared-linux-amd64',cf,CF_SHA)
 cf.chmod(0o700)
 server=spawn([python,comfy/'main.py','--listen','127.0.0.1','--port','8188','--lowvram','--force-fp16','--preview-method','none','--output-directory',OUT/'images','--disable-auto-launch'],OUT/'comfy.log')
 deadline=time.monotonic()+180
 while True:
  if server.poll() is not None:raise RuntimeError('ComfyUI stopped. See comfy.log in Kaggle outputs.')
  try:
   with urllib.request.urlopen('http://127.0.0.1:8188/object_info',timeout=3) as r:info=json.load(r)
   break
  except OSError:
   if time.monotonic()>deadline:raise RuntimeError('ComfyUI startup timed out. See comfy.log.')
   time.sleep(1)
 # Validate installed node classes before offering the browser link.
 for wf in WORKFLOW_PAYLOAD.values():
  for node in wf['nodes']:
   if node['type'] not in info:raise RuntimeError('Required node unavailable: '+node['type'])
 tunnel=spawn([cf,'tunnel','--url','http://127.0.0.1:8188','--no-autoupdate'],OUT/'tunnel.log')
 deadline=time.monotonic()+90
 url=None
 while time.monotonic()<deadline:
  text=(OUT/'tunnel.log').read_text()
  matches=re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com',text)
  if matches:url=matches[-1];break
  if tunnel.poll() is not None:raise RuntimeError('Cloudflare stopped; inspect tunnel.log.')
  time.sleep(1)
 if not url:raise RuntimeError('No Cloudflare URL received; inspect tunnel.log.')
 print('\nOPEN IMAGE STUDIO:',url,flush=True)
 print('Workflow menu → Image Studio: '+', '.join(WORKFLOW_PAYLOAD)+'. Select a workflow, then press Run.')
 print('No generation has started. Keep this cell running. Anyone with this session URL can access the studio.')
 print('Save Version with outputs before ending Kaggle. Images:',OUT/'images',flush=True)
 while server.poll() is None and tunnel.poll() is None:time.sleep(2)
 raise RuntimeError('Studio or tunnel stopped; see the saved logs.')
finally:
 for p in reversed(children):
  if p.poll() is None:
   os.killpg(p.pid,signal.SIGTERM)
   try:p.wait(timeout=10)
   except subprocess.TimeoutExpired:os.killpg(p.pid,signal.SIGKILL);p.wait()
 fcntl.flock(lock,fcntl.LOCK_UN);lock.close()
